# 02. Preparación experimental del dataset APTOS 2019

Este notebook construye las particiones experimentales utilizadas para
el entrenamiento, selección y evaluación de los modelos especializados
en Retinopatía Diabética.

Se utiliza exclusivamente el conjunto depurado y congelado durante la
auditoría previa, compuesto por 3487 retinografías.

Las particiones se generan de forma estratificada según el grado de
Retinopatía Diabética:

- Train: 70 %
- Validation: 15 %
- Test: 15 %

El conjunto Validation se utilizará para la comparación y selección de
las arquitecturas candidatas.

El conjunto Test permanecerá aislado y no será utilizado durante el
entrenamiento, ajuste de hiperparámetros ni selección del modelo.

## 1. Configuración del entorno, importaciones y rutas

Carga de librerías para manipulación tabular y división estratificada de datos (`scikit-learn`), definición de rutas del proyecto y semilla de reproducibilidad (`RANDOM_SEED = 42`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path(
    "/mnt/c/Users/SeuNg720/Documents/TP1-RetiScan"
)
APTOS_ROOT = (
    PROJECT_ROOT
    / "datasets"
    / "RD"
    / "aptos2019"
)
METADATA_DIR = (
    APTOS_ROOT
    / "metadata"
)
MASTER_PATH = (
    METADATA_DIR
    / "aptos2019_master_clean.csv"
)
RANDOM_SEED = 42

print(
    "Master:",
    MASTER_PATH
)
print(
    "Existe:",
    MASTER_PATH.exists()
)

Master: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/datasets/RD/aptos2019/metadata/aptos2019_master_clean.csv
Existe: True


## 2. Carga del conjunto maestro depurado

Lectura del archivo `aptos2019_master_clean.csv` generado al cierre de la auditoría y verificación de las dimensiones y distribución original de clases (3487 registros).

In [2]:
master_df = pd.read_csv(
    MASTER_PATH
)
print(
    "Registros:",
    len(master_df)
)
display(
    master_df.head()
)
print(
    "\nDistribución:"
)
print(
    master_df[
        "diagnosis"
    ]
    .value_counts()
    .sort_index()
)

Registros: 3487


,id_code,image_name,image_path,diagnosis,sha256
0,000c1434d8d7,000c1434d8d7.png,raw/train_images/000c1434d8d7.png,2,3f34b577a7102dddea20d2aaed76028bcf724ddb82e923...
1,001639a390f0,001639a390f0.png,raw/train_images/001639a390f0.png,4,9e4a10bcf0569bd7640ff3107601cb00c1147c44de5d3e...
2,0024cdab0c1e,0024cdab0c1e.png,raw/train_images/0024cdab0c1e.png,1,ef0bda2c21f3dbe839b4e5168fa7e52afe61e6923a56fd...
3,002c21358ce6,002c21358ce6.png,raw/train_images/002c21358ce6.png,0,c18a236541ae3d3ddbf8cf3eae991b4ff28446c5313b3d...
4,005b95c28852,005b95c28852.png,raw/train_images/005b95c28852.png,0,4af20410946f8730d1a9c13361a0b4604c5934c325890b...



Distribución:
diagnosis
0    1796
1     338
2     921
3     168
4     264
Name: count, dtype: int64


## 3. Generación de las particiones experimentales estratificadas

División en dos etapas con `train_test_split` estratificando por la columna `diagnosis`: primero se aísla el conjunto Test (15 %), y sobre el 85 % remanente se extrae el conjunto Validation (15 % del total, equivalente a `0.15 / 0.85`), dejando un 70 % para Train.

In [3]:
# Primero separamos Test (15%)
train_val_df, test_df = train_test_split(
    master_df,
    test_size=0.15,
    random_state=RANDOM_SEED,
    stratify=master_df["diagnosis"]
)

# Luego extraemos Validation del 85 % restante (15% del total global)
validation_relative_size = (
    0.15 / 0.85
)
train_df, validation_df = train_test_split(
    train_val_df,
    test_size=validation_relative_size,
    random_state=RANDOM_SEED,
    stratify=train_val_df["diagnosis"]
)

# Ordenamiento determinista por id_code
train_df = (
    train_df
    .sort_values("id_code")
    .reset_index(drop=True)
)
validation_df = (
    validation_df
    .sort_values("id_code")
    .reset_index(drop=True)
)
test_df = (
    test_df
    .sort_values("id_code")
    .reset_index(drop=True)
)

## 4. Verificación de dimensiones de las particiones

Comprobación de los tamaños absolutos de Train, Validation y Test para constatar que sumen exactamente las 3487 observaciones del dataset depurado.

In [4]:
print(
    "Train:",
    len(train_df)
)
print(
    "Validation:",
    len(validation_df)
)
print(
    "Test:",
    len(test_df)
)
print(
    "Total:",
    (
        len(train_df)
        + len(validation_df)
        + len(test_df)
    )
)

Train: 2440
Validation: 523
Test: 524
Total: 3487


## 5. Verificación de la distribución de clases por partición

Se comprueba que las particiones Train, Validation y Test conserven una
distribución proporcional de las cinco categorías diagnósticas respecto
al conjunto depurado original.

La estratificación busca evitar que las clases minoritarias queden
subrepresentadas en alguno de los subconjuntos experimentales.

In [5]:
def split_distribution(
    df,
    split_name
):
    result = (
        df["diagnosis"]
        .value_counts()
        .sort_index()
        .rename_axis("diagnosis")
        .reset_index(
            name="cantidad"
        )
    )
    result["porcentaje"] = (
        result["cantidad"]
        / len(df)
        * 100
    )
    result["split"] = split_name
    return result

train_distribution = split_distribution(
    train_df,
    "train"
)
validation_distribution = split_distribution(
    validation_df,
    "validation"
)
test_distribution = split_distribution(
    test_df,
    "test"
)

splits_distribution_df = pd.concat(
    [
        train_distribution,
        validation_distribution,
        test_distribution
    ],
    ignore_index=True
)
display(
    splits_distribution_df
)

,diagnosis,cantidad,porcentaje,split
0,0,1257,51.516393,train
1,1,236,9.672131,train
2,2,645,26.434426,train
3,3,118,4.836066,train
4,4,184,7.540984,train
5,0,269,51.434034,validation
6,1,51,9.751434,validation
7,2,138,26.386233,validation
8,3,25,4.780115,validation
9,4,40,7.648184,validation


## 6. Verificación de independencia entre particiones

Se comprueba que ningún identificador de imagen aparezca simultáneamente
en más de una partición experimental.

Esta verificación evita fuga directa de información entre Train,
Validation y Test.

In [6]:
train_ids = set(
    train_df["id_code"]
)
validation_ids = set(
    validation_df["id_code"]
)
test_ids = set(
    test_df["id_code"]
)

train_validation_overlap = (
    train_ids
    & validation_ids
)
train_test_overlap = (
    train_ids
    & test_ids
)
validation_test_overlap = (
    validation_ids
    & test_ids
)

print(
    "Train ∩ Validation:",
    len(train_validation_overlap)
)
print(
    "Train ∩ Test:",
    len(train_test_overlap)
)
print(
    "Validation ∩ Test:",
    len(validation_test_overlap)
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


### 6.1. Comprobación de cobertura total de identificadores

Verificación de que la unión de los tres subconjuntos sea idéntica y exhaustiva respecto a los 3487 registros de `master_df`.

In [7]:
all_split_ids = (
    train_ids
    | validation_ids
    | test_ids
)
print(
    "IDs únicos entre los 3 splits:",
    len(all_split_ids)
)
print(
    "IDs del master:",
    len(master_df)
)
print(
    "Cobertura completa:",
    all_split_ids
    == set(
        master_df["id_code"]
    )
)

IDs únicos entre los 3 splits: 3487
IDs del master: 3487
Cobertura completa: True


## 7. Aserciones de integridad del particionamiento

Validación formal mediante aserciones para garantizar que los tamaños, la ausencia de solapamiento, la cobertura total y la presencia de todas las clases se cumplan estrictamente.

In [8]:
assert len(train_df) == 2440
assert len(validation_df) == 523
assert len(test_df) == 524
assert len(
    train_df
) + len(
    validation_df
) + len(
    test_df
) == 3487
assert len(
    train_validation_overlap
) == 0
assert len(
    train_test_overlap
) == 0
assert len(
    validation_test_overlap
) == 0
assert all_split_ids == set(
    master_df["id_code"]
)
assert set(
    train_df["diagnosis"].unique()
) == {0, 1, 2, 3, 4}
assert set(
    validation_df["diagnosis"].unique()
) == {0, 1, 2, 3, 4}
assert set(
    test_df["diagnosis"].unique()
) == {0, 1, 2, 3, 4}
print(
    "Todas las aserciones del particionamiento fueron superadas."
)

Todas las aserciones del particionamiento fueron superadas.


## 8. Persistencia definitiva de las particiones experimentales

Una vez comprobada la estratificación, la ausencia de solapamientos y la
cobertura completa del conjunto depurado, las particiones Train,
Validation y Test se almacenan de forma permanente.

Estas particiones quedan congeladas para todos los experimentos posteriores.
Ambas arquitecturas candidatas utilizarán exactamente las mismas imágenes
en cada subconjunto.

El conjunto Test permanecerá aislado hasta la evaluación final del modelo
seleccionado.

In [9]:
train_final_df = train_df.copy()
validation_final_df = validation_df.copy()
test_final_df = test_df.copy()

train_final_df[
    "experimental_split"
] = "train"
validation_final_df[
    "experimental_split"
] = "validation"
test_final_df[
    "experimental_split"
] = "test"

splits_df = pd.concat(
    [
        train_final_df,
        validation_final_df,
        test_final_df
    ],
    ignore_index=True
)
splits_df = (
    splits_df
    .sort_values("id_code")
    .reset_index(drop=True)
)

print(
    "Registros del archivo de splits:",
    len(splits_df)
)
print(
    "\nDistribución por split:"
)
print(
    splits_df[
        "experimental_split"
    ].value_counts()
)

Registros del archivo de splits: 3487

Distribución por split:
experimental_split
train         2440
test           524
validation     523
Name: count, dtype: int64


### 8.1. Almacenamiento de archivos CSV en el directorio de metadatos

Se guardan en disco las particiones individuales (`aptos2019_train.csv`, `aptos2019_validation.csv`, `aptos2019_test.csv`) y el archivo maestro consolidado `aptos2019_splits.csv`.

In [10]:
SPLITS_PATH = (
    METADATA_DIR
    / "aptos2019_splits.csv"
)
TRAIN_SPLIT_PATH = (
    METADATA_DIR
    / "aptos2019_train.csv"
)
VALIDATION_SPLIT_PATH = (
    METADATA_DIR
    / "aptos2019_validation.csv"
)
TEST_SPLIT_PATH = (
    METADATA_DIR
    / "aptos2019_test.csv"
)

splits_df.to_csv(
    SPLITS_PATH,
    index=False
)
train_final_df.to_csv(
    TRAIN_SPLIT_PATH,
    index=False
)
validation_final_df.to_csv(
    VALIDATION_SPLIT_PATH,
    index=False
)
test_final_df.to_csv(
    TEST_SPLIT_PATH,
    index=False
)

print(
    "Splits guardados:"
)
print(
    "-",
    SPLITS_PATH
)
print(
    "-",
    TRAIN_SPLIT_PATH
)
print(
    "-",
    VALIDATION_SPLIT_PATH
)
print(
    "-",
    TEST_SPLIT_PATH
)

Splits guardados:
- /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/datasets/RD/aptos2019/metadata/aptos2019_splits.csv
- /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/datasets/RD/aptos2019/metadata/aptos2019_train.csv
- /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/datasets/RD/aptos2019/metadata/aptos2019_validation.csv
- /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/datasets/RD/aptos2019/metadata/aptos2019_test.csv


## 9. Registro de configuración del particionamiento

Se almacena la configuración utilizada para generar las particiones
experimentales, incluyendo la semilla aleatoria, proporciones y método
de estratificación.

In [11]:
import json

SPLIT_CONFIG_PATH = (
    METADATA_DIR
    / "aptos2019_split_config.json"
)

split_config = {
    "random_seed": RANDOM_SEED,
    "stratification_variable": "diagnosis",
    "split_strategy": "stratified",
    "requested_proportions": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    },
    "actual_counts": {
        "train": int(len(train_final_df)),
        "validation": int(len(validation_final_df)),
        "test": int(len(test_final_df))
    },
    "test_usage": "Final evaluation only"
}

with open(
    SPLIT_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        split_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Configuración guardada en:",
    SPLIT_CONFIG_PATH
)

Configuración guardada en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/datasets/RD/aptos2019/metadata/aptos2019_split_config.json


## 10. Verificación de los archivos guardados

Lectura independiente de los archivos CSV persistidos en disco para comprobar que su tamaño y contenido coinciden exactamente con las estructuras generadas en memoria.

In [12]:
saved_train_df = pd.read_csv(
    TRAIN_SPLIT_PATH
)
saved_validation_df = pd.read_csv(
    VALIDATION_SPLIT_PATH
)
saved_test_df = pd.read_csv(
    TEST_SPLIT_PATH
)
saved_splits_df = pd.read_csv(
    SPLITS_PATH
)

print(
    "Train guardado:",
    len(saved_train_df)
)
print(
    "Validation guardado:",
    len(saved_validation_df)
)
print(
    "Test guardado:",
    len(saved_test_df)
)
print(
    "Master splits:",
    len(saved_splits_df)
)

Train guardado: 2440
Validation guardado: 523
Test guardado: 524
Master splits: 3487


### 10.1. Aserciones finales de persistencia e integridad

Validación rigurosa mediante aserciones para verificar que los identificadores de cada split guardado en disco coincidan exactamente con los conjuntos en memoria y que las etiquetas del split sean únicamente `train`, `validation` y `test`.

In [13]:
assert len(
    saved_train_df
) == 2440
assert len(
    saved_validation_df
) == 523
assert len(
    saved_test_df
) == 524
assert len(
    saved_splits_df
) == 3487
assert set(
    saved_train_df["id_code"]
) == train_ids
assert set(
    saved_validation_df["id_code"]
) == validation_ids
assert set(
    saved_test_df["id_code"]
) == test_ids
assert set(
    saved_splits_df[
        "experimental_split"
    ].unique()
) == {
    "train",
    "validation",
    "test"
}
print(
    "Persistencia de particiones verificada correctamente."
)

Persistencia de particiones verificada correctamente.


## 11. Conclusión
Se generaron y congelaron las particiones experimentales del conjunto depurado APTOS 2019 mediante un particionamiento estratificado según el grado de Retinopatía Diabética.

El conjunto final de 3487 imágenes quedó distribuido en:

- Train: 2440 imágenes.
- Validation: 523 imágenes.
- Test: 524 imágenes.

Las verificaciones confirmaron ausencia de solapamiento entre particiones, cobertura completa del conjunto maestro y representación de las cinco categorías diagnósticas en cada subconjunto.

Las particiones fueron almacenadas de forma persistente y serán reutilizadas sin modificaciones en todos los experimentos posteriores, garantizando una comparación directa y reproducible entre las arquitecturas candidatas.

El conjunto Test queda reservado exclusivamente para la evaluación final del modelo seleccionado.